# Washer FSM — PYNQ-Z2 Test Bench

This notebook reads the VHDL FSM state register through **AXI GPIO / MMIO** on the **Zynq Processing System (PS)** and broadcasts state transitions to **WebSocket** clients (including the Godot/Dart frontend).

**Goal of this version:** split the original notebook into separate cells **without creating `.py` files or changing its behavior**. All functions and the server loop remain inside this notebook.

**Prerequisites:** run on the PYNQ-Z2 with a compatible overlay already loaded and the `pynq` and `websockets` packages available. Execute the cells in order. Previous execution outputs have been cleared.


## 1. Imports


In [1]:
import asyncio
import json

import websockets
from pynq import MMIO


## 2. AXI GPIO / MMIO

Initialize memory-mapped access to the state register on the PYNQ-Z2. The FPGA design must be loaded before running this cell.


In [2]:
# -----------------------------------
# AXI GPIO / MMIO
# -----------------------------------

BASE_ADDR = 0x41200000
ADDR_RANGE = 0x1000

mmio = MMIO(BASE_ADDR, ADDR_RANGE)


## 3. WebSocket Clients


In [3]:
# -----------------------------------
# WebSocket clients
# -----------------------------------

clients = set()


## 4. FSM State Decoding


In [4]:
# -----------------------------------
# FSM decoding
# -----------------------------------

STATE_MAP = {
    0: "IDLE",
    1: "FILL",
    2: "WASH",
    3: "RINSE",
    4: "SPIN",
    5: "DONE",
}


## 5. WebSocket Connection Handling

This function preserves the current behavior: it tracks connected clients and waits for their connections to close. It does not process incoming `START` / `RESET` commands.


In [5]:
# -----------------------------------
# WebSocket handler
# -----------------------------------

async def ws_handler(websocket):
    clients.add(websocket)

    print("Client connected")

    try:
        await websocket.wait_closed()

    finally:
        clients.remove(websocket)
        print("Client disconnected")


## 6. Broadcasting Updates to Clients


In [6]:
# -----------------------------------
# Broadcast helper
# -----------------------------------

async def broadcast(msg: str):
    dead = []

    for ws in clients:
        try:
            await ws.send(msg)
        except:
            dead.append(ws)

    for ws in dead:
        clients.remove(ws)


## 7. FSM Polling

Read the register; send a JSON message only when the decoded state changes.


In [1]:
# -----------------------------------
# FSM polling loop
# -----------------------------------

async def fsm_loop():

    previous_state = None

    while True:

        raw_value = mmio.read(0)

        state = STATE_MAP.get(raw_value, "UNKNOWN")

        if state != previous_state:

            print(f"FSM -> {state}")

            payload = {
                "type": "state",
                "value": state
            }

            await broadcast(json.dumps(payload))

            previous_state = state

        await asyncio.sleep(0.05)


## 8. Server Setup and Startup


In [8]:
# -----------------------------------
# Main
# -----------------------------------

async def main():

    server = await websockets.serve(
        ws_handler,
        "0.0.0.0",
        8765
    )

    print("WebSocket server running")

    asyncio.create_task(fsm_loop())

    await server.wait_closed()


## 9. Execution

Run the preceding cells in order, then run this cell to start the server. **As in the original notebook, this cell remains active** while the server is running. To stop it, interrupt the Jupyter kernel's execution. Explicit cleanup of asynchronous tasks can be addressed in a future step.


In [9]:
await main()

WebSocket server running
FSM -> IDLE
FSM -> FILL
FSM -> WASH
FSM -> RINSE
FSM -> SPIN
FSM -> DONE
FSM -> IDLE


CancelledError: 